<a href="https://colab.research.google.com/github/cpython-projects/da_27_07_2026/blob/main/lesson_16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лекція 16. Регулярні вирази

**Навіщо це потрібно.** Реальні дані майже завжди "бруднi" текстом: телефони в п'яти форматах, email з помилками, адреси в одному рядку, ціни зі вставленими пробілами. Регулярні вирази — це мова для опису текстових шаблонів, яка дозволяє все це знаходити, витягувати, очищати й перевіряти масово, а не руками.

**Регулярні вирази** — це формальна мова для опису шаблонів у тексті.

Використовуються для:

* **пошуку**
* **витягання**
* **заміни**
* **валідації** фрагментів тексту.

**Приклади аналітичних завдань:**

* Перевірити, що значення — email
* Витягти домен з email
* Очистити номер телефону від зайвих символів
* Знайти лише ті рядки, де згадується подія
* Витягти дату з рядка

In [1]:
import re

text = "Замовлення #4521 на суму 1200 грн, телефон +380671234567"

print(re.findall(r'\d+', text))       # усі послідовності цифр

['4521', '1200', '380671234567']


Один рядок коду — і ми витягли з довільного тексту всі числа, не описуючи вручну, де саме вони стоять. Саме так регулярні вирази й використовуються: замість того, щоб писати купу `if`/`split`/`replace` для кожного окремого формату тексту, ми одним шаблоном описуємо, *як виглядає* потрібний фрагмент, а `re`/`pandas` самі його знаходять.

Нижче — з яких елементів складається такий шаблон.

## БАЗОВІ ЕЛЕМЕНТИ РЕГУЛЯРНИХ ВИРАЗІВ

Регулярні вирази складаються з:

* **Символів-класів** (що шукати)
* **Квантифікаторів** (скільки разів)
* **Позиційних символів** (де саме)
* **Групування та логіки** (або, множини, групи)
* **Екранізації спецсимволів**

### I. Символи-класи (що шукати)

| Вираз | Назва     | Значення                         | Приклад                 |
| ----- | --------- | --------------------------------- | ------------------------ |
| `\d`  | digit     | цифра (0–9)                      | `\d+` — одна або більше |
| `\D`  | not digit | все, крім цифр                   | `\D+` — лише текст      |
| `\w`  | word      | літера (включно з Unicode), цифра або `_` | `\w+` — слово   |
| `\W`  | not word  | все, крім `\w`                   | `\W+` — пробіли, знаки  |
| `\s`  | space     | пробіл, табуляція, перенос рядка | `\s+` — роздільники     |
| `\S`  | not space | не пробіл                        | `\S+` — щільний текст   |

In [2]:
import re

print(re.findall(r'\w+', 'привіт Oleh_123'))

['привіт', 'Oleh_123']


### II. Позиційні символи (де шукати)

| Символ | Значення      | Приклад                          |
| ------ | -------------- | ---------------------------------- |
| `^`    | початок рядка | `^ID` — починається з "ID"       |
| `$`    | кінець рядка  | `.com$` — закінчується на ".com" |
| `\b`   | межа слова    | `\bcat\b` — лише слово "cat"     |
| `\B`   | не межа слова | `\Bcat\B` — всередині слова      |

In [3]:
import re

text = "cat catalog scatter"

print("findall(r'cat'):      ", re.findall(r'cat', text))
print("findall(r'\\bcat\\b'): ", re.findall(r'\bcat\b', text))

findall(r'cat'):       ['cat', 'cat', 'cat']
findall(r'\bcat\b'):  ['cat']


Без `\b` слово `cat` знаходиться усюди, де зустрічається ця послідовність букв — у тому числі всередині `catalog` і `scatter`. З `\b` з обох боків — лише там, де `cat` стоїть як окреме слово.

### III. Квантифікатори (скільки разів зустрічається)

| Вираз   | Значення             | Приклад                            |
| ------- | --------------------- | ------------------------------------ |
| `*`     | 0 або більше разів   | `\d*` — 0 і більше цифр            |
| `+`     | 1 або більше разів   | `\d+` — мінімум 1 цифра            |
| `?`     | 0 або 1 раз          | `s?` — літера s необов'язкова      |
| `{n}`   | рівно n разів        | `\d{4}` — 4 цифри (наприклад, рік) |
| `{n,}`  | від n разів і більше | `\d{2,}` — від двох цифр           |
| `{n,m}` | від n до m разів     | `\d{2,4}` — 2–4 цифри              |

### Жадібні (greedy) vs ліниві (lazy) квантифікатори

⚠️ Усі квантифікатори вище — **жадібні**: вони захоплюють максимально довгий збіг. Додавши `?` **після** квантифікатора (`*?`, `+?`, `??`, `{n,m}?`), робимо його **ліниво** — він захоплює мінімально можливий збіг.

> Зверніть увагу: тут `?` має **іншу** роль, ніж у рядку вище! Сам по собі `?` означає "0 або 1 раз", а `?` **після іншого квантифікатора** означає "зроби цей квантифікатор лінивим". Це типове джерело плутанини для початківців.

Різниця критична на тексті з кількома однаковими роздільниками — класичний приклад: видалення HTML-тегів (побачимо нижче в прикладах). Порівняємо на простому рядку:

In [4]:
import re

text = "<a>текст</a> ще <b>текст</b>"

greedy = re.findall(r'<.+>', text)      # жадібний
lazy = re.findall(r'<.+?>', text)       # ліниий

print("Жадібний (greedy):", greedy)
print("Ліниий (lazy):    ", lazy)

Жадібний (greedy): ['<a>текст</a> ще <b>текст</b>']
Ліниий (lazy):     ['<a>', '</a>', '<b>', '</b>']


Жадібний `<.+>` захоплює **все від першого `<` до останнього `>`** — тобто одним "тегом" проковтує і реальний текст між тегами! Ліниий `<.+?>` зупиняється на найближчому `>`, тож коректно знаходить кожен тег окремо. Для очищення HTML (та загалом для будь-яких "парних роздільників") майже завжди потрібен саме ліниий варіант.

### IV. Групування, альтернатива та множини

**1. Дужки `()` — групування**

Використовуються для створення **груп**, зручно з `str.extract()`:

* `(\d{4})-(\d{2})-(\d{2})` — витягає дату (рік, місяць, день)
* В pandas створюються окремі стовпці по групах

**2. Альтернатива `|` — або**

* `cat|dog|bird` — одне з слів
* `(http|https)://` — будь-який протокол (можна й коротше: `https?://`, бо `s` тут необов'язкова)

**3. Квадратні дужки `[]` — множини символів**

| Вираз    | Значення                    | Приклад             |
| -------- | ------------------------------ | --------------------- |
| `[abc]`  | один із символів a, b або c | `[aeiou]` — голосні |
| `[a-z]`  | діапазон                    | `[0-9]` — цифри     |
| `[^a-z]` | все, крім вказаних          | `[^0-9]` — не цифри |

### V. Екранізація спецсимволів (Escape)

Регулярні вирази використовують **спецсимволи**, які потрібно екранувати `\`, якщо хочеш шукати їх буквально:

| Символ                      | Призначення       |
| ------------------------------ | -------------------- |
| `\.`                        | крапка            |
| `\*`                        | зірочка           |
| `\\`                        | зворотний слеш    |
| `\+`, `\?`, `\(`, `\)` тощо | також екрануються |

**Приклад:**
`r"\.\w+"` — шукає крапку та слово після неї, наприклад: `.txt`, `.csv`.

## Приклади

### 1. **Символьні класи (попередньо визначені шаблони)**

| Регулярка | Що означає                | Приклад рядка                  | Знайде (`findall`)                                                                                                                         |
| ----------- | ---------------------------- | --------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------- |
| `\d`      | Цифра (0–9)               | `'Вартість: 1200 грн'`         | `'1'`, `'2'`, `'0'`, `'0'`                                                                                                                 |
| `\D`      | Не цифра                  | `'Вартість: 1200 грн'`         | `'В'`, `'а'`, `'р'`, `'т'`, `'і'`, `'с'`, `'т'`, `'ь'`, `':'`, `' '`                                                                       |
| `\w`      | Літера/цифра/підкреслення | `'user_id_42'`                 | `'u'`, `'s'`, `'e'`, `'r'`, `'_'`, `'i'`, `'d'`, `'_'`, `'4'`, `'2'`                                                                       |
| `\W`      | Все, крім `\w`            | `'user#id!'`                   | `'#'`, `'!'`                                                                                                                               |
| `\s`      | Пробільний символ         | `'item1\titem2  item3\nitem4'` | `'\t'`, `' '`, `' '`, `'\n'`                                                                                                               |
| `\S`      | Не пробіл                 | `'item1\titem2  item3\nitem4'` | `'i'`, `'t'`, `'e'`, `'m'`, `'1'`, `'i'`, `'t'`, `'e'`, `'m'`, `'2'`, `'i'`, `'t'`, `'e'`, `'m'`, `'3'`, `'i'`, `'t'`, `'e'`, `'m'`, `'4'` |

---

### 2. **Явні символи**

| Регулярка | Що означає                  | Приклад             | Результат                  |
| ----------- | ------------------------------- | ---------------------- | ----------------------------- |
| `.`       | Будь-який символ, крім `\n` | `'a1#b'`            | `'a'`, `'1'`, `'#'`, `'b'` |
| `\.`      | Буквально крапка            | `'v1.2 file.txt'`   | `'.'`, `'.'`               |
| `\\`      | Буквально зворотний слеш    | `'C:\\Users\\Oleh'` | `'\\'`, `'\\'`             |

---

### 3. **Квантифікатори (повтори)**

Тут важливо розрізняти `re.search()` (перший збіг) і `re.findall()` (усі збіги) — для `*`/`?` результат залежить від цього неочевидним чином.

In [5]:
import re

s = 'caaaat'

print("search('a*') =", re.search(r'a*', s))    # неочевидно!
print("search('a?') =", re.search(r'a?', s))    # неочевидно!
print("search('a+') =", re.search(r'a+', s))
print("findall('a*') =", re.findall(r'a*', s))
print("findall('a+') =", re.findall(r'a+', s))

search('a*') = <re.Match object; span=(0, 0), match=''>
search('a?') = <re.Match object; span=(0, 0), match=''>
search('a+') = <re.Match object; span=(1, 5), match='aaaa'>
findall('a*') = ['', 'aaaa', '', '']
findall('a+') = ['aaaa']


> ⚠️ **Пастка, у якій губляться новачки.** `a*` і `a?` дозволяють **0** повторень — тому `re.search()` знаходить збіг **одразу в позиції 0** (перед `'c'`), і це **порожній рядок** `''`, а не `'aaaa'`! Те саме з `a?`. Це одна з найчастіших причин, чому `str.extract()` у pandas несподівано повертає порожні рядки замість очікуваного тексту: якщо шаблон може збігтися з "нічим", він і збігається з "нічим" у першій же можливій позиції.
>
> Тому в таблиці нижче наведені коректні результати:

In [ ]:
print("re.search('a*', 'caaaat'):", re.search(r'a*', 'caaaat').group())   # '' (порожньо!)
print("re.search('a?', 'caaaat'):", re.search(r'a?', 'caaaat').group())   # '' (порожньо!)

| Регулярка | Опис             | Приклад    | Реальний `re.search()`        | Якщо потрібен саме непорожній збіг |
| ----------- | ------------------ | ------------ | --------------------------------- | ------------------------------------- |
| `a*`      | 0 або більше `a` | `'caaaat'` | `''` (порожньо, позиція 0!)     | Використати `a+`, або `findall` і відфільтрувати непорожні |
| `a+`      | 1 або більше `a` | `'caaaat'` | `'aaaa'`                        | — (тут усе гаразд, `a+` не допускає 0) |
| `a?`      | 0 або 1 `a`      | `'caaaat'` | `''` (порожньо, позиція 0!)     | Використати `a` без квантифікатора, якщо шукаєте саме наявність |
| `a{3}`    | Рівно 3 `a`      | `'aaaab'`  | `'aaa'`                         | — |
| `a{2,4}`  | Від 2 до 4 `a`   | `'aaaaaa'` | `'aaaa'`                        | — |
| `a{2,}`   | 2 і більше `a`   | `'aaab'`   | `'aaa'`                         | — |

---

### 4. **Групи та діапазони**

| Регулярка  | Що означає                    | Приклад рядка | Результат (`findall`)                                |
| ------------ | -------------------------------- | --------------- | -------------------------------------------------------- |
| `[abc]`    | Один символ: a, b або c       | `'ball'`      | `'b'`, `'a'`                                    |
| `[^abc]`   | Все, крім a, b, c             | `'dog'`       | `'d'`, `'o'`, `'g'`                              |
| `[a-z]`    | Будь-яка мала латинська буква | `'data2025'`  | `'d'`, `'a'`, `'t'`, `'a'`                     |
| `[0-9]`    | Цифри від 0 до 9              | `'v2025'`     | `'2'`, `'0'`, `'2'`, `'5'`                     |
| `[A-Za-z]` | Будь-яка латинська буква      | `'UserID'`    | `'U'`, `'s'`, `'e'`, `'r'`, `'I'`, `'D'`     |

---

### 5. **Групування та альтернативи**

| Регулярка | Що означає     | Приклад                   | Результат (`findall`)   |
| ----------- | ---------------- | ---------------------------- | -------------------------- |
| `(abc)`   | Група символів | `'abcabc'`                 | `'abc'`, `'abc'`         |

Приклади з альтернативою `\|` (без таблиці, щоб не зламати розмітку):

* `(cat\|dog\|bird)` — альтернатива. На рядку `'I have a dog and a cat'` знайде `'dog'`, `'cat'`.
* `(http\|https)://` — альтернатива для протоколу. На рядку `'https://site.com'` знайде `'https://'`.

---

### 6. **Якорі (межі)**

| Регулярка | Що означає    | Приклад            | Результат |
| ----------- | --------------- | -------------------- | ----------- |
| `^abc`    | Початок рядка | `'abcdef'`         | `'abc'`   |
| `abc$`    | Кінець рядка  | `'zzzabc'`         | `'abc'`   |
| `\\bcat\\b` | Слово цілком  | `'a cat in a hat'` | `'cat'`   |

---

# Приклади на даних

## 1. **Очистка телефонних номерів**

У датасеті є "сирі" номери:

```python
import pandas as pd

df = pd.DataFrame({
    "customer_id": [1, 2, 3, 4],
    "phone": ["+380 67 123-45-67", "0671234567", "38067123-45-67", "(067) 123 45 67"]
})
```

**Задача:** привести телефони до єдиного формату `+380XXXXXXXXX`.

In [ ]:
import pandas as pd

In [ ]:
df = pd.DataFrame({
    "customer_id": [1, 2, 3, 4],
    "phone": ["+380 67 123-45-67", "0671234567", "38067123-45-67", "(067) 123 45 67"]
})

In [ ]:
df

In [ ]:
df['phone'] = df.phone.str.replace(r'\D', '', regex=True)
df

In [ ]:
df['phone'] = df.phone.apply(lambda item: '+380' + item[-9:])
df

> ⚠️ **Важливе застереження.** Код вище спрацював, бо в усіх 4 номерах після очищення залишилось рівно 9 "локальних" цифр. Але є два реальні ризики, про які варто пам'ятати:
>
> 1. **`.apply(lambda item: ...)` на "сирому" рядку впаде на `NaN`.** Якщо хоч одне значення в колонці — пропуск (`NaN`, тип `float`), викличе `AttributeError`, бо в `NaN` немає методу `item[-9:]`. Безпечніший варіант — працювати через `.str`-акцесор pandas, який сам коректно пропускає `NaN`:
>
> ```python
> df['phone'] = '+380' + df.phone.str[-9:]
> ```
>
> 2. **Жодної перевірки результату немає.** Якщо в номері було на цифру більше чи менше (одрук, зайва цифра коду країни), код мовчки "приліпить" `+380` до неправильних 9 цифр і дасть **правдоподібний, але хибний** номер. Завжди варто валідувати фінальний формат:

In [ ]:
# NaN-безпечний варіант
df['phone_safe'] = '+380' + df.phone.str[-9:]

# перевірка: чи ВСІ номери відповідають очікуваному формату?
is_valid = df.phone_safe.str.fullmatch(r'\+380\d{9}')
print(is_valid.tolist())
print('Усі коректні:', is_valid.all())

### 2. **Виділення домену з email**

```python
df = pd.DataFrame({
    "email": ["user1@gmail.com", "manager@company.org", "test123@yahoo.com"]
})
```

**Задача:** отримати тільки домени (`gmail.com`, `company.org`...).

In [ ]:
df = pd.DataFrame({
    "email": ["user1@gmail.com", "manager@company.org", "test123@yahoo.com", "test56789@gmail"]
})
df

Можна й без regex — простим `.split()`. Але варто одразу через `.str.split()`, а не `.apply(lambda item: item.split(...))`, щоб не впасти на `NaN` (та саме застереження, що й вище з телефонами):

In [ ]:
df['email_domains'] = df.email.str.split('@').str[1]
df

Регулярками — те саме, але одразу двома колонками й стійко до `NaN`:

In [ ]:
df['email_domains'] = df.email.str.extract(r'@(.+)$')
df['email_users'] = df.email.str.extract(r'^(.+)@')
df

💡 **Корисно знати: іменовані групи.** Замість того, щоб витягувати колонки по одній і підписувати їх вручну, можна одразу назвати групи в самому регулярному виразі через `(?P<name>...)` — pandas сам використає ці назви як назви колонок:

In [ ]:
df[['email_users_named', 'email_domains_named']] = df.email.str.extract(
    r'^(?P<user>.+)@(?P<domain>.+)$'
).rename(columns={'user': 'email_users_named', 'domain': 'email_domains_named'})
df[['email', 'email_users_named', 'email_domains_named']]

## 3. **Фільтрація рядків по шаблону**

Є список замовлень із кодами товарів:

```python
df = pd.DataFrame({
    "order_id": [101, 102, 103, 104],
    "product_code": ["AB-12345", "CD-54321", "AB-67890", "XY-00000"]
})
```

**Задача:** знайти всі товари, що починаються на `AB-`.

In [ ]:
df = pd.DataFrame({
    "order_id": [101, 102, 103, 104, 105],
    "product_code": ["AB-12345", "CD-54321", "AB-67890", "XY-00000", "XB-12345"]
})
df

In [ ]:
df[df.product_code.str.startswith('AB-')]

In [ ]:
df[df.product_code.str.contains(r'^AB-')]

💡 **Коли regex, а коли ні.** Обидва варіанти вище дають однаковий результат — але `str.startswith()` **не є регулярним виразом**, а просте порівняння рядка, тож він швидший і однозначніший, коли потрібен саме префікс/суфікс. Regex (`str.contains` з `^...`) варто брати лише тоді, коли умова справді складніша за "починається/закінчується на" — наприклад, "починається на AB- **або** CD-, і далі рівно 5 цифр":

In [ ]:
# (?:...) - "незахоплююча" група: логіка "або" працює, але pandas не намагається
# повернути її як окрему колонку (без (?:...) str.contains() видає UserWarning
# про те, що в шаблоні є групи, і радить замість нього str.extract())
df[df.product_code.str.contains(r'^(?:AB|CD)-\d{5}$')]

💡 **Регістр не має значення?** Якщо коди товарів іноді вводять у нижньому регістрі (`ab-12345`), додайте `case=False` (для `contains`/`match`) або `flags=re.IGNORECASE` — працює для `contains`, `match`, `extract`, `replace`:

In [ ]:
import re

test_codes = pd.Series(["AB-12345", "ab-67890", "cd-11111"])
print(test_codes.str.contains(r'^ab-', case=False).tolist())
print(test_codes.str.contains(r'^ab-', flags=re.IGNORECASE).tolist())

## 4. **Парсинг тексту на кілька колонок**

У колонці `address` зберігається рядок:

```
"Kyiv, Khreshchatyk 12, 01001"
```

```python
df = pd.DataFrame({
    "address": [
        "Kyiv, Khreshchatyk 12, 01001",
        "Lviv, Svobody Ave 25, 79000"
    ]
})
```

**Задача:** розбити на `city`, `street`, `postal_code`.

In [ ]:
df = pd.DataFrame({
    "address": [
        "Kyiv, Khreshchatyk 12, 01001",
        "Lviv, Svobody Ave 25, 79000"
    ]
})
df

In [ ]:
df[['city', 'street', 'post_index']] = df.address.str.extract(r'^([^,]+),\s*([^,]+),\s*(\d+)$')
df

Або одразу з іменованими групами (без ручного перейменування колонок):

In [ ]:
df.address.str.extract(r'^(?P<city>[^,]+),\s*(?P<street>[^,]+),\s*(?P<post_index>\d+)$')

> ⚠️ **Найважливіше застереження для `str.extract()`.** Якщо рядок **не підходить** під шаблон (наприклад, адреса без індексу, або з трьома комами), `str.extract()` **не кидає помилку** — вона мовчки повертає `NaN` у всіх групах для цього рядка. Такі "тихі" пропуски легко не помітити. Завжди перевіряйте, скільки рядків не розпарсилось:

In [ ]:
df_check = pd.DataFrame({
    "address": [
        "Kyiv, Khreshchatyk 12, 01001",
        "Lviv, Svobody Ave 25, 79000",
        "Odesa Deribasivska 1",          # немає ком — не підійде під шаблон
    ]
})

parsed = df_check.address.str.extract(r'^(?P<city>[^,]+),\s*(?P<street>[^,]+),\s*(?P<post_index>\d+)$')
print(parsed)
print('Рядків, які НЕ розпарсились:', parsed.isna().any(axis=1).sum())

## 5. **Видалення HTML-тегів із тексту**

```python
df = pd.DataFrame({
    "review": [
        "<p>Good product!</p>",
        "<div>Average <b>quality</b></div>"
    ]
})
```

**Задача:** очистити текст від тегів.

In [ ]:
df = pd.DataFrame({
    "review": [
        "<p>Good product!</p>",
        "<div>Average <b>quality</b></div>"
    ]
})

df

In [ ]:
df['clean_review'] = df.review.str.replace(r'<.+?>', '', regex=True)
df

Саме тут і спрацьовує ліниий квантифікатор `+?` з початку лекції. Порівняйте, що було б із жадібним `<.+>`:

In [ ]:
df['clean_review_greedy_BAD'] = df.review.str.replace(r'<.+>', '', regex=True)
df[['review', 'clean_review', 'clean_review_greedy_BAD']]

Результат вражаючий: в **обох** рядках жадібний варіант видаляє геть усе до порожнього рядка — навіть "Good product!" з першого рядка зникає. Причина та сама в обох випадках: `.+` жадібно тягнеться від найпершого `<` у рядку до найостаннішого `>` у цьому ж рядку, тобто весь текст між ними (разом із реальним вмістом!) потрапляє всередину "одного тега". Це дуже поширена й неприємна помилка при чищенні HTML/XML — і працює вона тихо, без жодної помилки чи попередження, просто мовчки знищує текст.

## 6. **Виділення валюти та суми**

```python
df = pd.DataFrame({
    "price": ["USD 1200", "EUR 900", "UAH 45000"]
})
```

**Задача:** розбити на `currency` і `amount`.

In [ ]:
df = pd.DataFrame({
    "price": ["USD 1200", "EUR 900", "UAH       45 000"]
})

In [ ]:
df

In [ ]:
df[['currency', 'value']] = df.price.str.extract(r'^(\w+)\s*([\d ]+)$')
df

> ⚠️ **Важливий завершальний крок.** Регулярка витягла `value` як **текст**, і в третьому рядку там досі лишився пробіл усередині числа (`'45 000'`) — з таким рядком не можна рахувати (`sum()`, `mean()` впадуть). Регекс дав лише текстову заготовку; завжди потрібен завершальний крок приведення типу:

In [ ]:
df['value'] = df.value.str.replace(' ', '', regex=False).astype(float)
df

In [ ]:
df.dtypes

Тепер можна коректно рахувати, наприклад, суму по кожній валюті.

# Задачі

## 1. **Нормалізація номерів банківських карток**

У даних номери карток записані у різному вигляді.
**Задача:** привести їх до єдиного формату `1234 5678 9012 3456`.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "card_number": [
        "1234-5678-9012-3456",
        "1234 5678 9012 3456",
        "1234567890123456"
    ]
})

In [ ]:
df

In [ ]:
df['clean_card_number'] = df.card_number.str.replace(r'\D', '', regex=True)
df

In [ ]:
df['clean_card_number'] = df.clean_card_number.str.replace(r'([0-9]{4})(\d{4})(\d{4})(\d{4})', r'\1 \2 \3 \4', regex=True)
df

> 🔒 **Важливе застереження: дані карток — це не звичайний текст.** У реальній роботі номери банківських карток підпадають під стандарт **PCI DSS**, і зберігати чи показувати їх у повному вигляді (навіть у ноутбуці для навчання) — погана практика, яку не варто переносити в робочий код. На практиці аналітик майже завжди повинен **маскувати** номер, а не просто форматувати:

In [ ]:
# Маскування: показуємо лише останні 4 цифри
df['masked_card_number'] = df.clean_card_number.str.replace(
    r'(\d{4}) (\d{4}) (\d{4}) (\d{4})',
    r'**** **** **** \4',
    regex=True,
)
df[['card_number', 'masked_card_number']]

Саме такий (замаскований) вигляд даних про картки має сенс тримати в будь-якому звіті чи дашборді, якщо немає чіткого бізнес-дозволу та захищеного середовища для роботи з повними номерами.

## 2. **Виділення імені користувача з email**
**Задача:** отримати тільки ім'я користувача (до символа `@`).

In [ ]:
import pandas as pd
df = pd.DataFrame({
    "email": ["anna.petrenko@company.com", "ivan1990@gmail.com", "data_analyst@org.ua"]
})

In [ ]:
df

In [ ]:
df['username'] = df.email.str.extract(r'^(.+)@')
df

## 3. **Фільтрація за артикулом товару**
**Задача:** вибрати тільки ті товари, що мають код починається на `SKU-`.

In [ ]:
df = pd.DataFrame({
    "product_code": ["SKU-001-A", "SKU-002-B", "PRD-999-X", "SKU-003-C"]
})

In [ ]:
df[df.product_code.str.startswith('SKU-')]

## 4. **Аналіз одиниць виміру (вага)**

У даних вага вказана разом із одиницями виміру:
**Задача 1:** дізнатися, які саме одиниці виміру зустрічаються (і скільки разів кожна).
**Задача 2:** отримати числове значення окремо від одиниці.

In [ ]:
df = pd.DataFrame({
    "weight": ["12 kg", "800 g", "1.5 kg", "300 g", "2 t", "750 g", "0.5 kg"]
})

In [ ]:
df[['weight_new', 'unit']] = df.weight.str.extract(r'([\d\.]+)\s*(\D+)')
df

> 💡 Шаблон використовує `\s*` (0 або більше пробілів), а не буквальний пробіл — так він переживе й варіант без пробілу (`'12kg'`), якого в теперішніх даних немає, але який точно трапиться в реальних.

In [ ]:
df['weight_new'] = df.weight_new.astype('float')
df['unit'] = df.unit.str.strip().astype('category')
df

In [ ]:
df.unit.value_counts()

## 5. **Маркетингова аналітика: витягнути UTM-мітки з URL**

**Задача:** у вас лог відвідувань з повним URL. Потрібно витягнути `utm_source` і `utm_campaign`, щоб зрозуміти, звідки прийшов трафік — і окремо побачити, скільки заходів взагалі без UTM (прямий трафік чи забули проставити мітки).

In [ ]:
df = pd.DataFrame({
    "url": [
        "https://shop.ua/sale?utm_source=facebook&utm_campaign=autumn_sale&utm_medium=cpc",
        "https://shop.ua/sale?utm_source=google&utm_campaign=autumn_sale&utm_medium=cpc",
        "https://shop.ua/sale",
        "https://shop.ua/sale?utm_source=instagram&utm_campaign=black_friday",
    ]
})
df

In [ ]:
df[['utm_source', 'utm_campaign']] = df.url.str.extract(
    r'utm_source=(?P<utm_source>[^&]+).*?utm_campaign=(?P<utm_campaign>[^&]+)'
)
df

In [ ]:
print('Заходів без UTM-міток (прямий трафік або забули проставити):', df.utm_source.isna().sum())

> 💡 Зверніть увагу на `.*?` (лінивий квантифікатор) між двома параметрами — URL може містити параметри в будь-якому порядку і будь-якої довжини між `utm_source` і `utm_campaign`, а лінивий `.*?` не дає регексу "перестрибнути" через потрібний блок.

## 6. **Кілька збігів в одному рядку: `str.extractall`**

Раніше ми користувались лише `str.extract()`, який бере **перший** збіг. А що робити, якщо потрібних значень у рядку декілька — наприклад, хештеги в тексті поста?

In [ ]:
posts = pd.Series([
    "Люблю #python та #data_analysis!",
    "Сьогодні вивчаю #pandas, #regex і ще раз #python",
    "Просто пост без хештегів",
])

tags = posts.str.extractall(r'#(\w+)')
tags

Результат — таблиця з **мультиіндексом**: перший рівень — номер вихідного рядка, другий — порядковий номер збігу в цьому рядку. Це дозволяє одразу рахувати:

In [ ]:
print('Кількість хештегів у кожному пості:')
print(tags.groupby(level=0).size())

print()
print('Найпопулярніші хештеги загалом:')
print(tags[0].value_counts())

## 7. **Категоризація вільного тексту за ключовими словами**

Типова задача аналітика підтримки: відгуки чи тікети написані вільним текстом, але для звітності потрібні категорії. Regex + `np.select` дозволяють розкласти текст по категоріях без ручної розмітки кожного рядка.

In [ ]:
import numpy as np

reviews = pd.DataFrame({
    "text": [
        "Жахлива доставка, посилка йшла 3 тижні",
        "Чудовий товар, ціна супер",
        "Кур'єр нічого не сказав, доставка затрималась",
        "Якість жахлива, розірвалось за день",
        "Все ок, дякую",
    ]
})

conditions = [
    reviews.text.str.contains(r'доставк|кур\'єр|посилк', case=False, regex=True),
    reviews.text.str.contains(r'якіст|жахлив|розірв|поламал', case=False, regex=True),
    reviews.text.str.contains(r'ціна|дорог|дешев', case=False, regex=True),
]
choices = ['доставка', 'якість', 'ціна']

reviews['category'] = np.select(conditions, choices, default='інше')
reviews

> ⚠️ **Порядок умов у `np.select` важливий**: перша умова, що виконалась, і "виграє". Рядок 0 ("Жахлива доставка...") підпадає і під "доставка", і за словом "жахлива" теоретично міг би підпасти під "якість" — але оскільки умова "доставка" йде першою в списку, саме вона й присвоюється. На практиці це означає: розташовуйте найважливішу/найточнішу категорію першою.

## 8. **Парсинг лог-рядків**

Ще одна класична задача — розібрати рядки логів на структуровані поля (дата, час, рівень, повідомлення), щоб далі аналізувати частоту помилок, фільтрувати за рівнем тощо.

In [ ]:
logs = pd.Series([
    "2026-10-05 12:03:11 ERROR Failed to connect to db",
    "2026-10-05 12:03:15 INFO Request processed in 120ms",
    "2026-10-05 12:04:02 WARNING Slow query detected",
    "malformed log line without proper format",
])

parsed = logs.str.extract(
    r'(?P<date>\d{4}-\d{2}-\d{2}) (?P<time>\d{2}:\d{2}:\d{2}) (?P<level>\w+) (?P<message>.+)'
)
parsed

In [ ]:
print('Рядків, що НЕ відповідають формату логу:', parsed.date.isna().sum())
print()
print(parsed.level.value_counts())

> 💡 Останній рядок — не лог, а довільний текст. `str.extract()` не впав з помилкою, а мовчки поставив `NaN` у всі колонки — саме тому перевірка `.isna().sum()` після парсингу критична: без неї "зіпсований" рядок просто розчиниться в даних і зіпсує подальшу агрегацію.

## Підсумок: що ще варто знати аналітику про regex у pandas

| Інструмент | Що робить | Коли корисний |
|---|---|---|
| `.str.extract(pattern)` | Витягує **перший** збіг у кожному рядку, групи → колонки | Коли в рядку рівно одне потрібне значення |
| `.str.extractall(pattern)` | Витягує **всі** збіги в кожному рядку (мультиіндекс) | Коли в одному рядку може бути кілька збігів (кілька телефонів, хештегів тощо) |
| `(?P<name>...)` — іменовані групи | Автоматично підписує колонки назвами груп | Завжди, коли хочете одразу читабельні назви колонок без ручного перейменування |
| `.str.contains/match/fullmatch` | `contains` — шукає будь-де; `match` — тільки з початку рядка; `fullmatch` — весь рядок має відповідати | `fullmatch` — ідеальний інструмент для **валідації** формату (email, телефон, картка) |
| `case=False` / `flags=re.IGNORECASE` | Регістронезалежний пошук | Дані, де регістр не стандартизований |
| `.str.count(pattern)` | Рахує кількість збігів у кожному рядку | "Скільки цифр/слів/хештегів у рядку" |
| Ліниі квантифікатори (`+?`, `*?`) | Мінімальний, а не максимальний збіг | Парні роздільники (теги, дужки, лапки) |
| `.str` (а не `.apply(lambda x: x.метод())`) | Коректно пропускає `NaN` замість падіння з помилкою | Завжди, коли в колонці можуть бути пропуски |

### Три головні звички, які варто винести з цієї лекції

1. **Regex дає текст — завжди потрібен крок приведення типу** (`astype(float)`, прибирання пробілів тощо) перед тим, як рахувати статистики.
2. **`str.extract()` мовчки повертає `NaN`, якщо шаблон не підійшов** — завжди перевіряйте `.isna().any(axis=1).sum()` після парсингу, а не довіряйте "на око".
3. **Перевіряйте фінальний результат регуляркою-валідатором** (`str.fullmatch`), а не лише те, що код відпрацював без помилок — "відпрацювало" й "дало правильний результат" це різні речі.

### Завдання для самостійної роботи

1. У колонці `phone` з прикладу 1 додайте один рядок із заздалегідь неправильним номером (наприклад, на 2 цифри менше) і переконайтесь, що ваша валідація (`str.fullmatch`) це ловить.
2. У прикладі з UTM-мітками (п. 5) додайте витягання ще й `utm_medium`, і перевірте на URL, де параметри йдуть у іншому порядку.
3. У прикладі з адресами (п. 4) додайте рядок з адресою, що має зайву кому (наприклад, через номер квартири), і подивіться, що поверне `str.extract()`. Як виправити шаблон, щоб він все одно спрацював?
4. Перепишіть приклад з картками так, щоб маскована версія підтримувала і 15-значні картки (Amex), а не лише 16-значні.